# Andina Market Capa Gold (Star Schema)

Construye el modelo dimensional a partir de `lh_andina_market.silver.*` y lo
deja listo para el modelo semantico de Power BI (Direct Lake).

**Decisiones de diseño:**
- `dim_customer` implementa **SCD Tipo 2 real** via `MERGE` de Delta Lake —
  es el caso que el reto pide explicitamente (clientes que cambian de
  segmento). Las demas dimensiones (`dim_product`, `dim_date`) se
  recalculan por *full refresh* en cada corrida, por simplicidad — no
  requieren historizacion de atributos para el alcance de este reto.
- `fact_orders` hace un **join temporal** contra `dim_customer`
  (`OrderDate` entre `EffectiveDate` y `ExpirationDate`) para atribuir cada
  pedido al segmento/ciudad del cliente **vigente en ese momento**, no al
  actual.
- Grano de cada fact: `fact_orders` = 1 fila por pedido, `fact_order_items`
  = 1 fila por linea de pedido, `fact_payments` = 1 fila por pago,
  `fact_support_tickets` = 1 fila por ticket.


In [0]:
CATALOG = "andina_market"
SILVER = f"{CATALOG}.silver"
GOLD = f"{CATALOG}.gold"

## 1. dim_date

Calendario generado desde el rango real de fechas de negocio (con 7 dias de margen a cada lado), no hardcodeado.

In [0]:
from pyspark import pipelines as dp

@dp.materialized_view(name="andina_market.gold.dim_date")
def dim_date():
    return spark.sql("""
        WITH bounds AS (
            SELECT
                LEAST(MIN(OrderDate), MIN(CreatedAt))    AS min_date,
                GREATEST(MAX(OrderDate), MAX(CreatedAt)) AS max_date
            FROM andina_market.silver.orders
        ),
        calendar AS (
            SELECT explode(sequence(
                to_date(min_date) - INTERVAL 7 DAY,
                to_date(max_date) + INTERVAL 7 DAY,
                INTERVAL 1 DAY
            )) AS FullDate
            FROM bounds
        )
        SELECT
            CAST(date_format(FullDate, 'yyyyMMdd') AS INT) AS DateSK,
            FullDate,
            YEAR(FullDate)                                       AS Year,
            QUARTER(FullDate)                                    AS Quarter,
            MONTH(FullDate)                                       AS Month,
            date_format(FullDate, 'MMMM')                         AS MonthName,
            DAY(FullDate)                                         AS Day,
            date_format(FullDate, 'EEEE')                         AS DayName,
            CASE WHEN DAYOFWEEK(FullDate) IN (1, 7) THEN TRUE ELSE FALSE END AS IsWeekend
        FROM calendar
    """)

## 2. dim_product (SCD Tipo 1)

Surrogate key regenerada en cada corrida — valido porque este notebook hace *full refresh* de toda la capa gold en cada ejecucion (bronze/silver tambien se recalculan completos). Si mas adelante el pipeline pasa a ser incremental, este approach de SK necesitaria revisarse igual que se hizo con `dim_customer`.

In [0]:
@dp.materialized_view(name="dim_product")
def dim_product():
    return spark.sql("""
        SELECT
            ROW_NUMBER() OVER (ORDER BY ProductID) AS ProductSK,
            ProductID,
            SKU,
            ProductName,
            Category,
            UnitPrice AS CurrentUnitPrice,
            Status,
            had_invalid_price
        FROM andina_market.silver.products
    """)

## 3. dim_customer (SCD Tipo 2)

Patron estandar de Delta Lake en 2 pasos:
1. `MERGE` que **cierra** (marca `IsCurrent = FALSE`, setea `ExpirationDate`)
   la version vigente de cualquier cliente cuyos atributos cambiaron.
2. `INSERT` que agrega la **nueva version vigente** de esos clientes, mas
   los clientes completamente nuevos — con surrogate keys que continuan
   la secuencia existente (no se regeneran, para no romper las FKs que ya
   usan los facts de corridas anteriores).

In [0]:
@dp.materialized_view(name="dim_customer")
def dim_customer():
    return spark.sql("""
        SELECT
            ROW_NUMBER() OVER (ORDER BY CustomerID) AS CustomerSK,
            CustomerID,
            FullName,
            Email,
            Phone,
            City,
            Country,
            Segment,
            SignupDate,
            CURRENT_DATE() AS EffectiveDate,
            CAST(NULL AS DATE) AS ExpirationDate,
            TRUE AS IsCurrent
        FROM andina_market.silver.customers
    """)

## 4. fact_orders

Grano: 1 fila por pedido. El join contra `dim_customer` es **temporal**, no por `IsCurrent` — asi el pedido queda atribuido al segmento/ciudad que el cliente tenia *en la fecha del pedido*.

In [0]:
@dp.materialized_view(name="fact_orders")
def fact_orders():
    return spark.sql("""
        SELECT
            o.OrderID,
            dc.CustomerSK,
            CAST(date_format(o.OrderDate, 'yyyyMMdd') AS INT) AS OrderDateSK,
            o.Channel,
            o.Status,
            o.TotalAmount
        FROM andina_market.silver.orders o
        LEFT JOIN dim_customer dc
            ON o.CustomerID = dc.CustomerID
           AND CAST(o.OrderDate AS DATE) >= dc.EffectiveDate
           AND (dc.ExpirationDate IS NULL OR CAST(o.OrderDate AS DATE) < dc.ExpirationDate)
    """)

## 5. fact_order_items

Grano: 1 fila por linea de pedido.

In [0]:
@dp.materialized_view(name="andina_market.gold.fact_order_items")
def fact_order_items():
    return spark.sql("""
        SELECT
            oi.OrderItemID,
            fo.OrderID,
            dp.ProductSK,
            oi.Quantity,
            oi.UnitPrice,
            ROUND(oi.Quantity * oi.UnitPrice, 2) AS LineTotal,
            fo.OrderDateSK
        FROM andina_market.silver.order_items oi
        INNER JOIN fact_orders  fo ON oi.OrderID   = fo.OrderID
        INNER JOIN dim_product  dp ON oi.ProductID = dp.ProductID
    """)

## 6. fact_payments

Grano: 1 fila por pago (incluye reintentos, ya numerados en silver con `payment_attempt_number`).

In [0]:
@dp.materialized_view(name="andina_market.gold.fact_payments")
def fact_payments():
    return spark.sql("""
        SELECT
            p.PaymentID,
            fo.OrderID,
            fo.CustomerSK,
            CAST(date_format(p.PaymentDate, 'yyyyMMdd') AS INT) AS PaymentDateSK,
            p.PaymentMethod,
            p.Amount,
            p.Status,
            p.payment_attempt_number,
            p.amount_mismatch_flag
        FROM andina_market.silver.payments p
        INNER JOIN fact_orders fo ON p.OrderID = fo.OrderID
    """)

## 7. fact_support_tickets

Grano: 1 fila por ticket. Incluye `DaysToUpdate` como metrica base para un KPI de tiempo de resolucion en el Nivel 3.

In [0]:
@dp.materialized_view(name="andina_market.gold.fact_support_tickets")
def fact_support_tickets():
    return spark.sql("""
        SELECT
            t.TicketID,
            dc.CustomerSK,
            CAST(date_format(t.CreatedAt, 'yyyyMMdd') AS INT) AS CreatedDateSK,
            t.Status,
            t.Priority,
            DATEDIFF(t.UpdatedAt, t.CreatedAt) AS DaysToUpdate
        FROM andina_market.silver.support_tickets t
        LEFT JOIN dim_customer dc
            ON t.CustomerID = dc.CustomerID
           AND CAST(t.CreatedAt AS DATE) >= dc.EffectiveDate
           AND (dc.ExpirationDate IS NULL OR CAST(t.CreatedAt AS DATE) < dc.ExpirationDate)
    """)